# 03 Transformations and Actions

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Classify every common DataFrame operation as a <b>transformation</b> (lazy, returns a new DataFrame), an <b>action</b> (runs a job, returns a result) or <b>metadata only</b> (no job), and avoid the classic interview traps.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
"Is X a transformation or an action?" is asked in nearly every PySpark interview, often about tricky cases such as <code>cache</code>, <code>printSchema</code>, <code>show</code> or <code>spark.read</code>. The distinction also explains real behaviour: why an error appears three cells after the line that caused it, and why the same data is read twice.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Three kinds of operations</b>
<ul><li><b>Transformations</b> describe a new DataFrame from an existing one. They are <b>lazy</b>: Spark only records them in the plan</li><li><b>Narrow</b>: each output partition depends on one input partition (<code>select</code>, <code>filter</code>, <code>withColumn</code>, <code>union</code>). No shuffle</li><li><b>Wide</b>: output partitions depend on many input partitions (<code>groupBy</code>, <code>join</code>, <code>distinct</code>, <code>orderBy</code>, <code>repartition</code>). Shuffle</li><li><b>Actions</b> ask for a result, so Spark must execute the plan: they return data to the driver (<code>collect</code>, <code>count</code>, <code>show</code>, <code>take</code>) or write data out (<code>write</code>, <code>saveAsTable</code>)</li><li><b>Metadata operations</b> read the schema or plan only: <code>columns</code>, <code>dtypes</code>, <code>schema</code>, <code>printSchema</code>, <code>explain</code></li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A data engineer writes 40 lines of transformations, runs the cell, and it "finishes" in 0.1 seconds. They assume the data is loaded. Ten minutes later a <code>write</code> in another cell fails with a parsing error that actually comes from line 3. Nothing had run until the <code>write</code>, which was the first action.
</div>

## Classification table

| Operation | Kind | Notes |
|---|---|---|
| `select`, `withColumn(s)`, `drop`, `filter`/`where`, `union` | <span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> <span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> | No data movement |
| `groupBy().agg()`, `join` (non-broadcast), `distinct`, `orderBy`, `repartition` | <span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> <span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> | Shuffle |
| `coalesce(n)` | <span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> <span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> | Merges partitions without a full shuffle |
| `limit(n)`, `sample`, `dropDuplicates`, `describe()` | <span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> | Return DataFrames. `describe` computes when shown |
| `count`, `collect`, `show`, `display`, `take`, `first`, `head`, `toPandas`, `isEmpty` | <span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> | Return results to the driver |
| `write...save`, `saveAsTable`, `insertInto` | <span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> | Write data out |
| `foreach`, `foreachPartition` | <span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> | Run a function per row/partition on executors |
| `columns`, `dtypes`, `schema`, `printSchema`, `explain` | <span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata only</span> | No job |
| `cache` / `persist` | Lazy marker | Data is cached by the next action (not on serverless) |
| `spark.read...load()` | Mostly lazy | May run a job immediately to infer a schema or list files |

## 1. Transformations return instantly, actions do the work

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Times a chain of transformations and then the action that runs them.<br><br>
<b>Why it matters</b><br>Defining 10 transformations on 5 million rows takes milliseconds because nothing runs. The cost appears at the action.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Transformations take a few milliseconds. The <code>count()</code> action takes noticeably longer.
</div>

In [0]:
import time
from pyspark.sql import functions as F

start = time.time()
df = (
    spark.range(5_000_000)
    .withColumn("x", F.col("id") * 3)
    .filter(F.col("x") % 2 == 0)
    .withColumn("bucket", F.col("id") % 100)
    .groupBy("bucket").agg(F.sum("x").alias("total"))
)
print(f"transformations: {time.time() - start:.3f}s (nothing ran)")

start = time.time()
print("rows:", df.count())
print(f"action count():  {time.time() - start:.3f}s (the whole plan ran)")

## 2. Every transformation returns a new DataFrame

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows that a transformation leaves the original DataFrame unchanged and that the return types differ: transformations return <code>DataFrame</code>, actions return Python values.<br><br>
<b>Why it matters</b><br>A common bug is calling a transformation and ignoring its result, for example <code>df.filter(...)</code> on its own line, which changes nothing.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The original still has 5 rows after the "lost" filter. Then the types: <code>DataFrame</code> for transformations, <code>int</code>, <code>list</code> and <code>Row</code> for actions.
</div>

In [0]:
small = spark.range(5)
small.filter("id > 2")                # result ignored: small is unchanged
print("rows after ignored filter:", small.count())

filtered = small.filter("id > 2")     # keep the new DataFrame
print("rows in filtered:", filtered.count())

print("filter  ->", type(small.filter("id > 2")).__name__)
print("count   ->", type(small.count()).__name__)
print("collect ->", type(small.collect()).__name__)
print("first   ->", type(small.first()).__name__)

## 3. Metadata operations don't run jobs

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the columns, types and schema of a DataFrame whose <b>execution would fail</b>, to prove these calls never execute the data.<br><br>
<b>Why it matters</b><br><code>printSchema</code>, <code>columns</code> and <code>dtypes</code> only need the analyzed plan. They work even when running the plan would raise an error, which shows they don't run a job.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The schema prints fine. Only the <code>count()</code> raises a <code>DIVIDE_BY_ZERO</code> error.
</div>

In [0]:
broken = spark.range(10).withColumn("boom", F.lit(1) / (F.col("id") - F.col("id")))   # divides by zero on every row

print(broken.columns)
print(broken.dtypes)
broken.printSchema()

try:
    broken.count()
except Exception as e:
    print("count() executed the plan and failed:", type(e).__name__)

## 4. Interview traps

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Checks four operations that people often misclassify: <code>show</code>, <code>cache</code>, <code>describe</code> and reading files.<br><br>
<b>Why it matters</b>
<ul><li><code>show()</code> is an <b>action</b> (it runs a job to fetch 20 rows)</li><li><code>cache()</code> is <b>lazy</b>: it marks the DataFrame. The next action fills the cache. On serverless it isn't supported at all</li><li><code>describe()</code> returns a <b>DataFrame</b> (lazy). It computes only when shown</li><li><code>spark.read</code> is mostly lazy, but with <code>inferSchema=True</code> (or JSON without a schema) Spark runs a job immediately to look at the data</li></ul>
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>describe</code> returns a DataFrame instantly. <code>cache</code> either returns instantly (classic) or reports that caching isn't supported (serverless).
</div>

In [0]:
d = spark.range(1_000_000).describe("id")
print("describe returned:", type(d).__name__, "(nothing computed yet)")
d.show()                                              # now it computes

try:
    start = time.time()
    c = spark.range(1_000_000).withColumn("y", F.col("id") * 2).cache()
    print(f"cache() returned in {time.time() - start:.3f}s: it only marks the DataFrame")
    c.count()                                         # this action fills the cache
    c.unpersist()
except Exception as e:
    print("cache is not supported on this compute:", type(e).__name__)

## 5. Errors surface at the action

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines a transformation that references a valid column but produces invalid values, and shows the error only at the action.<br><br>
<b>Why it matters</b><br>Name errors (missing columns) are caught when the plan is analyzed. Data errors (bad casts, division by zero) are caught only when the job runs. When debugging, look <b>upstream</b> of the line that failed.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
"defined without error", then the action raises a <code>CAST_INVALID_INPUT</code> error.
</div>

In [0]:
raw = spark.createDataFrame([("1",), ("two",)], "v STRING")
casted = raw.withColumn("v_int", F.col("v").cast("int"))
print("defined without error")

try:
    casted.collect()
except Exception as e:
    print("error at the action:", getattr(e, "getCondition", lambda: type(e).__name__)())

## Under the hood

```
transformation  ->  adds a node to the LOGICAL PLAN (driver only, microseconds)
action          ->  analyze -> optimize -> physical plan -> DAG of stages -> tasks on executors
metadata        ->  analyze only (resolve names and types), no physical execution
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> builds the plan. <span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> executes it. <span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata only</span> reads it.

Because every action starts from the source again, two actions on the same DataFrame run the plan twice. The plan below is what any action on `df` would execute.

In [0]:
df.explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: "my filter didn't work"</b><br>
The transformation's result wasn't assigned. Write <code>df = df.filter(...)</code>.<br><br>
<b>⛔ Problem: a cell that "loads" data finishes instantly</b><br>Only transformations ran. Nothing is read until an action.<br><br>
<b>⛔ Problem: an error appears far from its cause</b><br>Data errors are raised at the action. Check the transformations that feed it.<br><br>
<b>⛔ Problem: data is read twice</b><br>Two actions on the same DataFrame. Combine them or materialize the result once.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between a transformation and an action?</b><br>
A transformation describes a new DataFrame and is lazy: Spark only adds it to the plan. An action asks for a result, such as returning rows or writing data, which triggers optimization and execution of the whole plan as a job.<br><br>

<b>🎤 2. Give examples of narrow and wide transformations.</b><br>
Narrow: <code>select</code>, <code>filter</code>, <code>withColumn</code>, <code>union</code>, <code>coalesce</code>, which don't move data between partitions. Wide: <code>groupBy</code>, <code>join</code>, <code>distinct</code>, <code>orderBy</code>, <code>repartition</code>, which need a shuffle.<br><br>

<b>🎤 3. Is <code>show()</code> an action?</b><br>
Yes. It runs a job to compute and fetch the first rows.<br><br>

<b>🎤 4. Is <code>cache()</code> an action?</b><br>
No. It's lazy: it marks the DataFrame for caching, and the next action materializes it. On Databricks serverless it isn't supported.<br><br>

<b>🎤 5. Is <code>printSchema()</code> an action?</b><br>
No. It only reads the analyzed plan's schema, so no job runs.<br><br>

<b>🎤 6. Can <code>spark.read</code> trigger a job?</b><br>
Yes. Reading with schema inference (CSV <code>inferSchema</code>, or JSON without a schema) runs a job immediately to sample the data. With an explicit schema it is lazy.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which of these triggers the execution of a Spark job?</b><br>
A. <code>df.withColumn("x", lit(1))</code><br>
B. <code>df.filter("amount &gt; 0")</code><br>
C. <code>df.write.saveAsTable("t")</code><br>
D. <code>df.select("id")</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b> Writing is an action. The others are lazy transformations.</details><br><br>

<b>Q2. Which operation is a wide transformation?</b><br>
A. <code>filter</code><br>
B. <code>select</code><br>
C. <code>groupBy().count()</code><br>
D. <code>withColumnRenamed</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Classify each as transformation (narrow/wide), action or metadata: <code>distinct</code>, <code>take(3)</code>, <code>dtypes</code>, <code>union</code>, <code>foreach</code>, <code>orderBy</code>, <code>limit</code>, <code>toPandas</code>, <code>explain</code></li><li>Prove with timing that <code>orderBy</code> alone runs nothing</li><li>Create a DataFrame whose action fails but whose <code>printSchema()</code> works</li><li>Explain in one sentence why <code>df.count(); df.show()</code> reads the source twice</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. distinct: wide transformation | take(3): action | dtypes: metadata | union: narrow transformation
#    foreach: action | orderBy: wide transformation | limit: transformation | toPandas: action | explain: metadata

# 2. orderBy alone is instant
start = time.time()
ordered = spark.range(5_000_000).orderBy(F.desc("id"))
print(f"orderBy defined in {time.time() - start:.3f}s")

# 3. Schema works, action fails
bad = spark.createDataFrame([("x",)], "s STRING").withColumn("n", F.col("s").cast("int"))
bad.printSchema()
try:
    bad.show()
except Exception as e:
    print("action failed:", type(e).__name__)

# 4. Each action is a separate job that executes the plan from the source, because nothing is materialized in between.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Transformations are lazy and return DataFrames. Actions run jobs and return results or write data</li><li>Narrow transformations don't shuffle. Wide ones do</li><li>Metadata calls (<code>columns</code>, <code>schema</code>, <code>printSchema</code>, <code>explain</code>) never run jobs</li><li><code>cache</code> is lazy (and unavailable on serverless). <code>show</code> is an action. <code>describe</code> returns a DataFrame</li><li><code>spark.read</code> with inference runs a job right away</li><li>Name errors appear at analysis. Data errors appear at the action</li></ul>
</div>

| Kind | Examples |
|---|---|
| Narrow | `select`, `filter`, `withColumn`, `union`, `coalesce` |
| Wide | `groupBy`, `join`, `distinct`, `orderBy`, `repartition` |
| Action | `count`, `collect`, `show`, `take`, `first`, `toPandas`, `write`, `foreach` |
| Metadata | `columns`, `dtypes`, `schema`, `printSchema`, `explain` |

## Git commit

```
git add 02_spark_internals/03_transformations_actions.ipynb
git commit -m "add 02_03 transformations actions notebook"
```